In [2]:
%pip install PyMuPDF



Note: you may need to restart the kernel to use updated packages.


In [3]:
import pickle
from pathlib import Path

CACHE_PATH = Path("../data/document_groups.pkl")

def load_document_groups():
    with open(CACHE_PATH, "rb") as f:
        return pickle.load(f)


In [4]:
data = load_document_groups()
print(f"Loaded {len(data)} applicants")
print(type(data))

Loaded 4 applicants
<class 'list'>


In [5]:
for group in data:
    print("Applicant:", group["applicant_id"])
    print("Zip:", group["zip_name"])
    print("Docs:", len(group["documents"]))
    print("Filenames:",
          [doc["filename"] for doc in group["documents"]])
    print("---")


Applicant: 1
Zip: Sample data files/GUL20240146-20241011T000608Z-001 (1).zip
Docs: 28
Filenames: ['GUL20240146/DISBURSEMENT MEMO_48.pdf', 'GUL20240146/OCR DOCUMENTS_28.pdf', 'GUL20240146/INSURANCE FORMS_44.pdf', 'GUL20240146/TECHNICAL REPORTS_65.pdf', 'GUL20240146/DRAFT SALE DEED - VETTED_63.pdf', 'GUL20240146/LEGAL APPRAISAL REPORT_66.pdf', 'GUL20240146/MOTD CHALLAN AND DRAFT COPY - VETTED_38.pdf', 'GUL20240146/MAIL CONFIRMATION FROM LEGAL OFFICERS_32.pdf', 'GUL20240146/OCR DOCUMENTS_78.pdf', 'GUL20240146/VENDOR KYC AND BANK DETAILS_40.pdf', 'GUL20240146/DISBURSEMENT MEMO_6.pdf', 'GUL20240146/LOAN OFFER LETTER FINAL_61.pdf', 'GUL20240146/NACH AND PDC_21.pdf', 'GUL20240146/VENDOR KYC AND BANK DETAILS_26.pdf', 'GUL20240146/OCR DOCUMENTS_50.pdf', 'GUL20240146/DISBURSEMENT MEMO_43.pdf', 'GUL20240146/TECHNICAL REPORTS_68.pdf', 'GUL20240146/DISBURSEMENT REQUEST FORM_31.pdf', 'GUL20240146/DEVIATION APPROVAL MAILS_34.pdf', 'GUL20240146/DISBURSEMENT REQUEST FORM_47.pdf', 'GUL20240146/LPN_13.pd

In [6]:
import fitz
from PIL import Image
import io
from typing import Generator, Optional

def pdf_bytes_to_images(pdf_bytes: bytes, dpi: int = 300, max_pages: Optional[int] = None) -> Generator[Image.Image, None, None]:
    """Yield PIL.Image pages from PDF bytes.

    - Handles real PDFs via PyMuPDF (`fitz`).
    - If the bytes are already a single-image file (JPEG/PNG/etc), tries to open with PIL and yield it.
    - Yields images lazily to avoid loading all pages simultaneously.

    Args:
        pdf_bytes: raw file bytes (PDF or image).
        dpi: rasterization DPI for PDF pages.
        max_pages: optional maximum pages to yield.
    """
    # Try opening as PDF first
    try:
        with fitz.open(stream=pdf_bytes, filetype="pdf") as doc:
            zoom = dpi / 72
            matrix = fitz.Matrix(zoom, zoom)
            for i in range(len(doc)):
                if max_pages is not None and i >= max_pages:
                    break
                page = doc.load_page(i)
                pix = page.get_pixmap(matrix=matrix, alpha=False)
                # Convert to PIL Image without copying large intermediate files
                img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
                yield img
                # allow pix to be freed
                del pix
            return
    except Exception:
        # Not a PDF or pdf parsing failed — try to open as an image
        pass

    # Fallback: try PIL open from bytes (JPEG/PNG/GIF/...)
    try:
        img = Image.open(io.BytesIO(pdf_bytes)).convert("RGB")
        yield img
    except Exception as e:
        raise RuntimeError(f"Unable to decode bytes as PDF or image: {e}")



In [7]:
def stream_pdf_pages(dpi: int = 300, max_pages_per_doc: int | None = None, only_pdf: bool = False):
    """Yield pages for all PDF (or image) documents in the cached document groups.

    Args:
        dpi: rasterization DPI for PDFs.
        max_pages_per_doc: if set, limits pages yielded per document.
        only_pdf: if True, skip non-PDF documents; if False, attempt to decode image bytes too.
    """
    data = load_document_groups()

    for group in data:
        applicant_id = group["applicant_id"]

        for doc in group["documents"]:
            filename = doc.get("filename", "")
            pdf_bytes = doc.get("content")
            if pdf_bytes is None:
                continue

            # If only_pdf is True, skip non-PDF filenames quickly
            if only_pdf and not filename.lower().endswith(".pdf"):
                continue

            try:
                # pdf_bytes_to_images is a generator — iterate lazily
                for page_num, img in enumerate(pdf_bytes_to_images(pdf_bytes, dpi=dpi, max_pages=max_pages_per_doc), start=1):
                    yield {
                        "applicant_id": applicant_id,
                        "filename": filename,
                        "page": page_num,
                        "image": img,
                    }

            except Exception as e:
                print(f"Conversion failed [{filename}]: {e}")


In [8]:
for item in stream_pdf_pages():
    print(item["applicant_id"], item["filename"], item["page"])
    # preprocess item["image"] → tensor → ManTraNet inference


1 GUL20240146/DISBURSEMENT MEMO_48.pdf 1
1 GUL20240146/DISBURSEMENT MEMO_48.pdf 2
1 GUL20240146/OCR DOCUMENTS_28.pdf 1
1 GUL20240146/INSURANCE FORMS_44.pdf 1
1 GUL20240146/OCR DOCUMENTS_28.pdf 1
1 GUL20240146/INSURANCE FORMS_44.pdf 1
1 GUL20240146/INSURANCE FORMS_44.pdf 2
1 GUL20240146/TECHNICAL REPORTS_65.pdf 1
1 GUL20240146/TECHNICAL REPORTS_65.pdf 2
1 GUL20240146/TECHNICAL REPORTS_65.pdf 3
1 GUL20240146/INSURANCE FORMS_44.pdf 2
1 GUL20240146/TECHNICAL REPORTS_65.pdf 1
1 GUL20240146/TECHNICAL REPORTS_65.pdf 2
1 GUL20240146/TECHNICAL REPORTS_65.pdf 3
1 GUL20240146/DRAFT SALE DEED - VETTED_63.pdf 1
1 GUL20240146/DRAFT SALE DEED - VETTED_63.pdf 2
1 GUL20240146/LEGAL APPRAISAL REPORT_66.pdf 1
1 GUL20240146/LEGAL APPRAISAL REPORT_66.pdf 2
1 GUL20240146/DRAFT SALE DEED - VETTED_63.pdf 1
1 GUL20240146/DRAFT SALE DEED - VETTED_63.pdf 2
1 GUL20240146/LEGAL APPRAISAL REPORT_66.pdf 1
1 GUL20240146/LEGAL APPRAISAL REPORT_66.pdf 2
1 GUL20240146/LEGAL APPRAISAL REPORT_66.pdf 3
1 GUL20240146/MOTD C

In [ ]:
import os
from pathlib import Path
import pandas as pd

IMAGES_DIR = Path("data/images")
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

def build_image_dataset(save_dir=IMAGES_DIR, dpi=300):
    records = []

    for group in load_document_groups():
        applicant_id = group["applicant_id"]
        applicant_dir = save_dir / f"applicant_{applicant_id}"
        applicant_dir.mkdir(exist_ok=True)

        for doc in group["documents"]:
            filename = doc["filename"]
            if not filename.lower().endswith(".pdf"):
                continue

            # Label must come from PKL structure or a lookup
            label = int(doc["real"])  # adjust to your schema
            pdf_bytes = doc["content"]

            images = pdf_bytes_to_images(pdf_bytes, dpi=dpi)

            for page_idx, img in enumerate(images):
                img_name = f"{Path(filename).stem}_page{page_idx}.png"
                img_path = applicant_dir / img_name

                img.save(img_path)

                records.append({
                    "applicant_id": applicant_id,
                    "doc_id": filename,
                    "page_idx": page_idx,
                    "image_path": str(img_path),
                    "label": label,
                })

    df = pd.DataFrame(records)
    return df


In [12]:
df = build_image_dataset()
df.head()



KeyError: 'is_real'

In [9]:
import torch
import torchvision.transforms as T
from PIL import ImageOps, Image

def preprocess_for_mantranet(image: Image.Image):
    # Convert to grayscale (most weights expect single-channel)
    image = image.convert("L")

    # Maintain aspect ratio via padding to square canvas
    w, h = image.size
    max_dim = max(w, h)
    padded = ImageOps.pad(image, (max_dim, max_dim), color=255)  # white pad

    # Resize to ManTraNet resolution (512x512)
    transform = T.Compose([
        T.Resize((512, 512)),
        T.ToTensor(),  # scales to [0,1], shape: (1, 512, 512)
    ])
    
    tensor = transform(padded).unsqueeze(0)  # add batch dimension
    return tensor


In [10]:
for item in stream_pdf_pages():
    tensor = preprocess_for_mantranet(item["image"]).to(device)

    # model output placeholder — depends on your architecture
    with torch.no_grad():
        output = model(tensor)

    print(item["applicant_id"], item["filename"], item["page"], "✓")


NameError: name 'device' is not defined

In [ ]:
import torch
import numpy as np
import torchvision.transforms as T
from PIL import Image, ImageOps

# --- Preprocessing ---

def _pad_to_square(image: Image.Image, fill: int = 255) -> Image.Image:
    """Pad a PIL image to a square canvas without distortion."""
    w, h = image.size
    if w == h:
        return image

    if w > h:
        pad_top = (w - h) // 2
        pad_bottom = w - h - pad_top
        padding = (0, pad_top, 0, pad_bottom)  # (left, top, right, bottom)
    else:
        pad_left = (h - w) // 2
        pad_right = h - w - pad_left
        padding = (pad_left, 0, pad_right, 0)

    return ImageOps.expand(image, border=padding, fill=fill)


_preprocess_size = (512, 512)

# Default preprocessing pipeline:
# - pad to square, resize to _preprocess_size
# - optional augmentations (only when `augment=True`)
# - convert to tensor and scale to 0-255 (ManTraNet's IMTFE divides by 255 internally)
def _build_preprocess(augment: bool = False):
    aug_steps = []
    if augment:
        aug_steps = [
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(degrees=5, resample=Image.BILINEAR),
        ]

    steps = [
        *aug_steps,
        T.Resize(_preprocess_size, interpolation=Image.BILINEAR),
        T.ToTensor(),  # -> [C, H, W] in [0,1]
        # We intentionally do NOT normalize here because ManTraNet's IMTFE
        # expects raw 0-255 inputs; the model code divides by 255 and scales to [-1,1].
    ]
    return T.Compose(steps)


def preprocess_for_mantranet(image: Image.Image, augment: bool = False) -> torch.Tensor:
    """Convert a PIL.Image page into a tensor ready for ManTraNet.

    Output shape: [1, 1, H, W] where H,W == `_preprocess_size`.
    The resulting tensor is in range [0, 255] (float32).
    """
    # Ensure grayscale single-channel
    image = image.convert("L")

    # Pad to square without distortion
    image = _pad_to_square(image, fill=255)

    # Build pipeline and apply
    pipeline = _build_preprocess(augment=augment)
    tensor = pipeline(image)  # [C, H, W], where C==1

    # Scale to 0-255 as ManTraNet expects
    tensor = tensor * 255.0

    # Add batch dim: [1, C, H, W]
    return tensor.unsqueeze(0).to(torch.float32)


In [ ]:
# Smoke test
for item in stream_pdf_pages(dpi=300):
    tensor = preprocess_for_mantranet(item["image"])
    print(tensor.shape)  # should be [1, 1, 512, 512]
    break


torch.Size([1, 1, 512, 512])


In [ ]:
def postprocess_heatmap(
    heatmap_tensor: torch.Tensor,
    heatmap_threshold: float = 0.5
):
    """
    Convert a raw heatmap tensor to:
      - heatmap: np.ndarray in [0,1], shape [H, W]
      - mask: boolean np.ndarray [H, W] of anomalous pixels
      - score: float, fraction of anomalous pixels

    heatmap_tensor: expected shape [1, 1, H, W] or [1, H, W].
    """
    # Remove batch / channel dims safely
    ht = heatmap_tensor.detach().cpu().float().squeeze()
    heatmap = ht.numpy()

    # Normalize defensively to [0,1]
    # (If your model already outputs [0,1], this is basically a noop.)
    min_val = heatmap.min()
    max_val = heatmap.max()
    if max_val > min_val:
        heatmap = (heatmap - min_val) / (max_val - min_val)
    else:
        heatmap = np.zeros_like(heatmap)

    mask = heatmap > heatmap_threshold
    score = mask.mean().item() if hasattr(mask.mean(), "item") else float(mask.mean())

    return heatmap, mask, score


In [ ]:
import torch
from pathlib import Path
from src.mantranet.model import ManTraNet

MODEL_PATH = Path("models/mantranet_splicing_v1.pth")

def load_mantranet_model(device="cpu"):
    model = ManTraNet()  # exact architecture that matches weights
    state_dict = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model


In [ ]:
# class ManTraNetInference:
#     def __init__(
#         self,
#         model: torch.nn.Module,
#         heatmap_threshold: float = 0.5,
#         anomaly_score_threshold: float = 0.01,
#         dpi: int = 300,
#     ):
#         """
#         model: a PyTorch ManTraNet-like model returning a per-pixel heatmap.
#         heatmap_threshold: threshold on per-pixel score to mark anomaly.
#         anomaly_score_threshold: threshold on fraction of anomalous pixels
#                                  to flag the page as suspicious.
#         dpi: rasterization DPI for PDF -> image.
#         """
#         self.model = model
#         self.model.eval()
#         self.heatmap_threshold = heatmap_threshold
#         self.anomaly_score_threshold = anomaly_score_threshold
#         self.dpi = dpi

#     @torch.no_grad()
#     def infer_page(self, page_image: Image.Image):
#         """
#         Run ManTraNet on a single PIL page image.
#         Returns dict with:
#           - heatmap (np.ndarray)
#           - mask (np.ndarray[bool])
#           - anomaly_score (float)
#           - is_suspicious (bool)
#         """
#         x = preprocess_for_mantranet(page_image)  # [1,1,512,512], on CPU
#         output = self.model(x)  # expected [1,1,H,W] or [1,H,W]

#         heatmap, mask, score = postprocess_heatmap(
#             output, heatmap_threshold=self.heatmap_threshold
#         )
#         is_suspicious = score >= self.anomaly_score_threshold

#         return {
#             "heatmap": heatmap,
#             "mask": mask,
#             "anomaly_score": score,
#             "is_suspicious": is_suspicious,
#         }

#     def run_over_stream(self):
#         """
#         Iterate over your entire dataset using stream_pdf_pages(),
#         yielding per-page results.

#         Yields:
#           {
#             "applicant_id": int,
#             "filename": str,
#             "page": int,
#             "anomaly_score": float,
#             "is_suspicious": bool,
#             "heatmap": np.ndarray,
#             "mask": np.ndarray[bool],
#           }
#         """
#         for item in stream_pdf_pages(dpi=self.dpi):
#             res = self.infer_page(item["image"])
#             yield {
#                 "applicant_id": item["applicant_id"],
#                 "filename": item["filename"],
#                 "page": item["page"],
#                 "anomaly_score": res["anomaly_score"],
#                 "is_suspicious": res["is_suspicious"],
#                 "heatmap": res["heatmap"],
#                 "mask": res["mask"],
#             }


main 

In [ ]:
import torch

# 1) Load your pretrained ManTraNet model here
# model = YourManTraNet()
# state = torch.load("mantranet_pretrained.pth", map_location="cpu")
# model.load_state_dict(state)
# model.eval()

# TEMP: placeholder to avoid NameError while you’re wiring things
class DummyModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = torch.nn.Conv2d(1, 1, kernel_size=3, padding=1)

    def forward(self, x):
        return torch.sigmoid(self.conv(x))

model = DummyModel()

# 2) Build inference engine
engine = ManTraNetInference(
    model=model,
    heatmap_threshold=0.5,
    anomaly_score_threshold=0.01,
    dpi=300,
)

# 3) Run over a few pages as a smoke test
for i, result in enumerate(engine.run_over_stream()):
    print(
        result["applicant_id"],
        result["filename"],
        "page", result["page"],
        "| score =", round(result["anomaly_score"], 4),
        "| suspicious =", result["is_suspicious"],
    )
    if i >= 5:
        break


In [ ]:
import torch
from pathlib import Path

# path where you cloned the repo
MANTRANET_DIR = Path("../models/mantranet")  # update if needed
import sys; sys.path.append(str(MANTRANET_DIR))

from mantranet import ManTraNet  # or exact class name found in mantranet.py


weights_path = Path("ManTraNet-pytorch/models/mantranetv4.pt")

model = ManTraNet()
state_dict = torch.load(weights_path, map_location="cpu")
model.load_state_dict(state_dict)
model.eval()
print("Model loaded successfully")

for item in stream_pdf_pages(dpi=200):  # throttle DPI for quick test
    tensor = preprocess_for_mantranet(item["image"])
    with torch.no_grad():
        output = model(tensor)
    print("OK:", item["filename"], "page", item["page"], "| output shape =", output.shape)
    break
